In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.sql("Use Data.Silver")

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 5, Finished, Available, Finished, False)

DataFrame[]

In [4]:
#config


RUN_MODE = "train_and_score"
MODEL_DIR = "Files/models/breach_lr_model"
RISK_TABLE = "gold_fact_ticket_scored"
CALIB_TABLE = "gold_breach_risk_calibration"


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 6, Finished, Available, Finished, False)

In [5]:
SILVER_TICKETS = "tickets"
SILVER_CUSTOMERS = "silver_customers"
SILVER_AGENTS="silver_agents"
SILVER_SLA="silver_policies"
SILVER_COMMENTS="silver_comments"
GOLD_FACT="Fact_tickets"


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 7, Finished, Available, Finished, False)

In [6]:
from pyspark.sql import functions as F, types as T, Window as W
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator




try:
    dbutils
    HAS_DBUTILS = True
except:
    HAS_DBUTILS = False




StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 8, Finished, Available, Finished, False)

In [7]:
print("[INFO] Loading silver tables....")
t =  spark.table(SILVER_TICKETS)
c = spark.table(SILVER_CUSTOMERS).select("customer_id","segment","region","mrr","is_vip")
a0 = spark.table(SILVER_AGENTS).select("agent_id","team","region","seniority")
a = a0.withColumnRenamed("region","agent_region")
sla=spark.table(SILVER_SLA).select("sla_id","priority","first_response_hours","resolution_hours")

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 9, Finished, Available, Finished, False)

[INFO] Loading silver tables....


In [8]:
print("[INFO] Aggregating comment sentiment")


com = spark.table(SILVER_COMMENTS).groupBy("ticket_id").agg(F.avg("sentiment_score").alias("Average sentiment"),F.count("*").alias("Comment Count"))



StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 10, Finished, Available, Finished, False)

[INFO] Aggregating comment sentiment


In [9]:
print("[INFO] Joining features...")

t_ = t.alias("t")
p_ = sla.alias("sla")

f = t_.join(c, "customer_id","left").join(a, "agent_id", "left").join(p_, "sla_id", "left").drop(p_["priority"]).join(com, "ticket_id","left")

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 11, Finished, Available, Finished, False)

[INFO] Joining features...


In [10]:
dow_sun1 = F.dayofweek(F.col("created_at"))
created_dow = ((dow_sun1 + 5 ) % 7) + 1



f = (
    f.withColumn("label_breach", F.col("breach_resolution").cast("int"))
     .withColumn("created_hour", F.hour("created_at"))
     .withColumn("created_dow", created_dow.cast("int"))
     .withColumn("is_weekend", (F.col("created_dow") >= 6).cast("int"))
)


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 12, Finished, Available, Finished, False)

In [11]:
#Avg proxy  per Customer



cust_activity = t.groupBy("customer_id").agg(F.count("*").alias("cust_total_tickets"))

f=f.join(cust_activity,"customer_id", "left").fillna({"Average sentiment":0.0, "Comment Count":0.0,"cust_total_tickets":0 })

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 13, Finished, Available, Finished, False)

In [12]:
f= f.withColumn("first_response_hours", F.col("first_response_hours").cast("int"))
f= f.withColumn("resolution_hours", F.col("resolution_hours").cast("int"))



StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 14, Finished, Available, Finished, False)

In [13]:
display(f)

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 80abf653-899c-429e-af64-bcdc2e1013a5)

In [14]:

features_df = f.select(
    "ticket_id","label_breach",
    "priority","channel","segment","region","team","seniority",
    "mrr","is_vip","created_hour","is_weekend","Average sentiment","Comment Count",
    "first_response_hours","resolution_hours","cust_total_tickets"
)


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 16, Finished, Available, Finished, False)

In [15]:
display(features_df)

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43294f78-ca7f-48a6-9743-e3142a1212ce)

In [16]:
#Training 

if RUN_MODE != "score_only":
    print("[INFO] Training Logistic Regression model..")
cat_cols = ["priority","channel","segment","region","team","seniority"]
num_cols = ["mrr","is_vip","created_hour","is_weekend","Average sentiment","Comment Count","first_response_hours","resolution_hours","cust_total_tickets"]


indexers = [StringIndexer(inputCol=c,outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]


encoders = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_oh"] ) for c in cat_cols]


assembler = VectorAssembler(inputCols=[f"{c}_oh" for c in cat_cols] + num_cols, outputCol="features_raw")

scaler    = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="label_breach", maxIter=50, regParam=0.01)

pipeline = Pipeline(stages=indexers + encoders + [assembler,scaler,lr])

train,test = features_df.randomSplit([0.8,0.2],seed=42)
model = pipeline.fit(train)
pred = model.transform(test)




auc = BinaryClassificationEvaluator(labelCol="label_breach",metricName="areaUnderROC").evaluate(pred)
print(f"[METRIC] Test AUC:{auc:.4f}")



StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 18, Finished, Available, Finished, False)

[INFO] Training Logistic Regression model..


[METRIC] Test AUC:0.9315


In [17]:

print("Catalog:", spark.catalog.currentCatalog())
print("Database:", spark.catalog.currentDatabase())


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 19, Finished, Available, Finished, False)

Catalog: spark_catalog
Database: chimcobldhq2alqj5l36irj1dphmaba4clr2ah31ehgiakr9dhr6asg


In [19]:
   # Save model to /Files (guard dbutils)
   


  

# create the folder (if needed) and save



workspace_id   = "13f1dba3-87b9-4ed6-87ec-70457f4c142a"  # NOT the name
lakehouse_name = "Data"


MODEL_DIR = f"abfss://13f1dba3-87b9-4ed6-87ec-70457f4c142a@onelake.dfs.fabric.microsoft.com/02d83474-9ff7-40a5-a7ab-81ac9b54080e/Files/models/breach_lr_model"
model.write().overwrite().save(MODEL_DIR)







StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 21, Finished, Available, Finished, False)

In [23]:
from pyspark.ml.pipeline import PipelineModel



MODEL_DIR = "/Files/models/breach_lr_model"
model = PipelineModel.load("abfss://13f1dba3-87b9-4ed6-87ec-70457f4c142a@onelake.dfs.fabric.microsoft.com/02d83474-9ff7-40a5-a7ab-81ac9b54080e/Files/models/breach_lr_model")


from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array


scored = (model.transform(features_df)
          .select(
              F.col("ticket_id"),
              vector_to_array(F.col("probability")).getItem(1).alias("BreachRisk")
          )
          .withColumnRenamed("ticket_id", "TicketId"))





StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 25, Finished, Available, Finished, False)

In [24]:
spark.sql("USE Data.Gold")

StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 26, Finished, Available, Finished, False)

DataFrame[]

In [26]:

scored.write.mode("overwrite").format("delta").saveAsTable("gold_fact_ticket_scored")
print(" Wrote: gold_fact_ticket_scored")


StatementMeta(, 0e4d5728-4dea-459e-b837-1a421ee14665, 28, Finished, Available, Finished, False)

 Wrote: gold_fact_ticket_scored
